### Membuat SparkSession

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, row_number
from pyspark.sql.window import Window
import numpy as np
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession aktif, versi Spark:", spark.version)

SparkSession aktif, versi Spark: 3.5.9


### Menyiapkan Dataset

In [15]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/xiuviu/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/xiuviu/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/xiuviu/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/xiuviu/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### Membaca dan Eksplorasi Awal

In [16]:
path_hdfs = "hdfs://localhost:9000/user/xiuviu/tugas5/transaksi_tugas5.csv"

df_transaksi = spark.read.csv(path_hdfs, header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.printSchema()
df_transaksi.show(5)
df_target.show()

root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



### Join

In [17]:
ringkasan_kota = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner").withColumn("pencapaian_persen", spark_round(col("total_pendapatan") / col("target_bulanan") * 100, 2))

hasil_a.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+-----------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen|
+----------+----------------+--------------+----------+-----------------+
| Purworejo|        45650000|      30000000|     Fitri|           152.17|
|      Solo|        33475000|      40000000|      Bayu|            83.69|
|Yogyakarta|        47275000|      60000000|      Joko|            78.79|
|  Magelang|        31650000|      45000000|      Rani|            70.33|
|  Semarang|        38175000|      55000000|      Sari|            69.41|
+----------+----------------+--------------+----------+-----------------+



### Window function

In [19]:
per_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))

window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())
hasil_b = per_kota_kategori.withColumn("urutan", row_number().over(window_kota)).filter(col("urutan") == 1).orderBy("kota")

hasil_b.show()

+----------+--------------------+----------------+------+
|      kota|            kategori|total_pendapatan|urutan|
+----------+--------------------+----------------+------+
|  Magelang|Kesehatan & Kecan...|         7275000|     1|
| Purworejo|Kesehatan & Kecan...|        10075000|     1|
|  Semarang|        Rumah Tangga|        11125000|     1|
|      Solo|Kesehatan & Kecan...|         8425000|     1|
|Yogyakarta|             Fashion|        13325000|     1|
+----------+--------------------+----------------+------+



### Spark sql

In [21]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

hasil_c = spark.sql('''
    SELECT t.kota, p.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang p ON t.kota = p.kota
    GROUP BY t.kota, p.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



### Kesimpulan
Purworejo berkinerja paling baik. Cabang ini mencatat pendapatan Rp45.650.000 terhadap target Rp30.000.000, sehingga pencapaiannya 152,17% dan menjadi satu-satunya cabang yang melampaui target. Purworejo juga memiliki transaksi terbanyak, 116 transaksi, dengan kategori terlaris Kesehatan & Kecantikan sebesar Rp10.075.000. Target Purworejo paling rendah di antara lima cabang, dan itu ikut menaikkan persentasenya.
Semarang paling perlu perhatian manajemen. Pencapaiannya 69,41% (Rp38.175.000 dari target Rp55.000.000), terendah di antara semua cabang, dengan selisih Rp16.825.000 dari target. Magelang menyusul di 70,33% (Rp31.650.000 dari Rp45.000.000). Yogyakarta membukukan pendapatan absolut tertinggi, Rp47.275.000, tetapi targetnya Rp60.000.000 sehingga pencapaiannya 78,79%. Solo mencapai 83,69%. Manajemen dapat mulai dari Semarang dan Magelang dengan memperkuat kategori terlaris masing-masing: Rumah Tangga di Semarang (Rp11.125.000) dan Kesehatan & Kecantikan di Magelang (Rp7.275.000).